# RBV2 ver32 — Policy Replay Analysis v3

מחברת זו מריצה replay על תמונות Raw מסשן קיים בעזרת ה־C++ CLI, ואז מוסיפה שכבת מדיניות ניסיונית מעל תוצאות המודל.

המדיניות החדשה כוללת:

- `Heuristics` בצהוב עבור weak bunch anchor עם תמיכת child tomatoes.
- `Color Correction` לעגבנייה בודדת כאשר צבע הסגמנטציה סותר את מחלקת הבשלות של המודל.
- `Review` עבור חיזויים חלשים איכותיים שנפלו קרוב לסף.

ה־Heuristics לא הופך Weak ל־Strong. הוא מוסיף תוצאה נפרדת למחקר חזותי וכמותי.


In [ ]:
from pathlib import Path
import json, subprocess, sys

PROJECT_ROOT = Path('/home/ronen/Desktop/RBV2_NJOrin_ver32_devFarm_tomato')
SESSION_PATH = PROJECT_ROOT / 'Debugging/toClient/sessions/session_20260630_103628'
MODE = 'policy_only'   # policy_only / robot_like
MAX_IMAGES = 0         # 0 = all images

SESSION_ID = SESSION_PATH.name
CPP_OUT = PROJECT_ROOT / 'policy_replay_lab/outputs' / SESSION_ID / 'cpp_replay'
ANALYSIS_OUT = PROJECT_ROOT / 'policy_replay_lab/outputs' / SESSION_ID / 'analysis'
CONFIG_PATH = PROJECT_ROOT / 'policy_replay_lab/configs/policy_v32_experiment.json'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('SESSION_PATH =', SESSION_PATH)
print('CPP_OUT =', CPP_OUT)
print('ANALYSIS_OUT =', ANALYSIS_OUT)


In [ ]:
# Build the C++ replay CLI.
subprocess.run(['bash', 'policy_replay_lab/scripts/build_policy_replay_cli.sh'], cwd=PROJECT_ROOT, check=True)


In [ ]:
# Run TensorRT replay on images_ok_raw and images_weak_noise_raw.
cmd = [
    './policy_replay_lab/bin/policy_replay_cli',
    '--project-root', '.',
    '--session', str(SESSION_PATH.relative_to(PROJECT_ROOT)),
    '--mode', MODE,
    '--output', str(CPP_OUT.relative_to(PROJECT_ROOT)),
    '--engine', 'models/best8s_seg_v43_fp16.engine',
    '--groups', 'images_ok_raw,images_weak_noise_raw',
    '--max-images', str(MAX_IMAGES),
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)


In [ ]:
# Add Heuristics layer and create CSV + visual debug images.
sys.path.insert(0, str(PROJECT_ROOT / 'policy_replay_lab/python'))
from policy_analysis import run_analysis

summary = run_analysis(
    cpp_jsonl=CPP_OUT / 'cpp_replay_detections.jsonl',
    config_path=CONFIG_PATH,
    output_dir=ANALYSIS_OUT,
    max_debug_images=0,   # 0 = draw all frames
)
print(json.dumps(summary, indent=2, ensure_ascii=False))

print('\nVisual debug folders:')
for k in ['old_policy_debug_dir', 'new_policy_debug_dir', 'heuristics_only_debug_dir', 'comparison_debug_dir']:
    print(f'{k}: {summary.get(k)}')
print('debug_image_index_csv:', summary.get('debug_image_index_csv'))


In [ ]:
# Quick table view.
try:
    import pandas as pd
    df = pd.read_csv(ANALYSIS_OUT / 'policy_compare.csv')
    display(df.groupby(['source_type', 'new_status']).size().reset_index(name='count'))
    display(df[df['source_type'] == 'heuristic'].groupby(['class_name', 'dominant_maturity']).size().reset_index(name='count'))
except Exception as e:
    print('Pandas display skipped:', e)
    print('CSV path:', ANALYSIS_OUT / 'policy_compare.csv')


In [ ]:
# Show a few side-by-side visual examples directly inside the notebook.
# Preference: frames that actually contain yellow Heuristics boxes.
try:
    import pandas as pd
    from IPython.display import display, Image
    idx = pd.read_csv(ANALYSIS_OUT / 'debug_image_index.csv')
    examples = idx[idx['heuristics_count'] > 0].head(5)
    if examples.empty:
        print('No Heuristics frames were created. Showing first comparison frames instead.')
        examples = idx.head(5)
    for _, row in examples.iterrows():
        print('frame_id=', row['frame_id'], ' image=', row['image_path'], ' heuristics_count=', row['heuristics_count'])
        display(Image(filename=str(row['comparison_debug_path'])))
except Exception as e:
    print('Visual display skipped:', e)
    print('Open images manually under:', ANALYSIS_OUT / 'comparison_debug')


## Outputs

המחברת שומרת גם קבצים מספריים וגם תמונות חזותיות עם BBOX.

- `policy_compare.csv` — השוואה מלאה: old_status מול new_status, כולל Review, Color Correction ו־Heuristics.
- `heuristics.jsonl` — רק תוצאות Heuristics בצהוב.
- `color_corrections.csv/jsonl` — חיזויי עגבנייה בודדת שתוקנו לפי צבע הסגמנטציה.
- `regular_review_candidates.csv/jsonl` — חיזויי Weak איכותיים לבדיקת Review.
- `manual_review_queue.csv` — כל מה ששווה בדיקה ידנית: Heuristics, Review, Color Correction.
- `summary.json` — סיכום ספירות ונתיבי פלט.
- `debug_image_index.csv` — אינדקס לכל תמונה מצוירת.
- `old_policy_debug/` — תמונות מקור עם BBOX לפי המדיניות הישנה בלבד.
- `new_policy_debug/` — אותן תמונות עם BBOX של המדיניות החדשה.
- `heuristics_only_debug/` — רק BBOX צהובים של Heuristics.
- `comparison_debug/` — השוואת old/new side-by-side.
